In [17]:
import joblib
import tarfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

import warnings
warnings.filterwarnings("ignore")

In [2]:
SEED = 66

In [3]:
df_full = pd.read_csv("cocktail_dataset.csv")

print(df_full.head(5))
print(df_full.columns)
print(df_full.shape)

   recipe_id  abbott's_bitters_pct  absinthe_pct  advocaat_liqueur_pct  \
0          1                   0.0           0.0                   0.0   
1          2                   0.0           0.0                   0.0   
2          3                   0.0           0.0                   0.0   
3          4                   0.0           0.0                   0.0   
4          5                   0.0           0.0                   0.0   

   agave_syrup_pct  aged_jamaican_rum_pct  aged_rum_(6-10yr)_pct  \
0              0.0                    0.0                    0.0   
1              0.0                    0.0                    0.0   
2              0.0                    0.0                    0.0   
3              0.0                    0.0                    0.0   
4              0.0                    0.0                    0.0   

   agricole_rhum_(unaged)_pct  almond_milk_liqueur_pct  amaretto_liqueur_pct  \
0                         0.0                      0.0            

In [4]:
flavor_categories = ['bittersweet', 'citrus', 'creamy', 'floral', 'fruity', 'herbal', 'savoury', 'spicy', 'sweet']
has_flavor_tags = df_full[flavor_categories].sum(axis=1) > 0
df = df_full[has_flavor_tags].copy()

print(df.head(5))
print(df.columns)
print(df.shape)

   recipe_id  abbott's_bitters_pct  absinthe_pct  advocaat_liqueur_pct  \
0          1                   0.0           0.0                   0.0   
1          2                   0.0           0.0                   0.0   
5          6                   0.0           0.0                   0.0   
7          8                   0.0           0.0                   0.0   
9         10                   0.0           0.0                   0.0   

   agave_syrup_pct  aged_jamaican_rum_pct  aged_rum_(6-10yr)_pct  \
0              0.0                    0.0                    0.0   
1              0.0                    0.0                    0.0   
5              0.0                    0.0                    0.0   
7              0.0                    0.0                    0.0   
9              0.0                    0.0                    0.0   

   agricole_rhum_(unaged)_pct  almond_milk_liqueur_pct  amaretto_liqueur_pct  \
0                         0.0                      0.0            

In [5]:
# Identify ingredient columns (all columns ending with '_pct')
ingredient_columns = [col for col in df.columns if col.endswith('_pct')]
print(f"Ingredient columns found: {len(ingredient_columns)}")

Ingredient columns found: 242


In [6]:
# Separate features (ingredients) and targets (flavors)
X = df[ingredient_columns].copy()
y = df[flavor_categories].copy()

print(f"Features: {X.shape}")
print(f"Targets: {y.shape}")

Features: (2692, 242)
Targets: (2692, 9)


In [7]:
# We check for NaN values
print(f"NaN values in features: {X.isna().values.sum()}")
print(f"NaN values in targets: {y.isna().values.sum()}")

total_len = len(y)

print("\n Flavor Distribution:")
for flavor in flavor_categories:
    count = y[flavor].sum()
    percentage = (count / total_len) * 100
    print(f"  {flavor:<12}: {percentage:.2f}% ({count} recipes)")

NaN values in features: 0
NaN values in targets: 0

 Flavor Distribution:
  bittersweet : 18.05% (486 recipes)
  citrus      : 35.55% (957 recipes)
  creamy      : 4.38% (118 recipes)
  floral      : 4.49% (121 recipes)
  fruity      : 27.60% (743 recipes)
  herbal      : 16.46% (443 recipes)
  savoury     : 3.42% (92 recipes)
  spicy       : 5.61% (151 recipes)
  sweet       : 5.50% (148 recipes)


In [14]:
# For multi-label data, we need to ensure stratified split
# Create a string key for each unique flavor combination
flavor_combo = y[flavor_categories].astype(str).agg(''.join, axis=1)

# Some flavor combos may appear very rarely (< 3 times), which causes
# stratified split to fail. Bucket those into a single 'rare' group.
combo_counts = flavor_combo.value_counts()
rare_combos = combo_counts[combo_counts < 3].index
stratify_col = flavor_combo.where(~flavor_combo.isin(rare_combos), other='rare')

print(f"Unique flavor combos: {flavor_combo.nunique()}")
print(f"Combos with < 3 samples (bucketed as 'rare'): {len(rare_combos)}")
print(f"Stratification groups: {stratify_col.nunique()}")

# Perform stratified split — class ratios will be preserved
X_train, X_test, y_train, y_test = train_test_split(
    X, y[flavor_categories],
    test_size=0.2,
    random_state=42,
    stratify=stratify_col
)

print(f"\nStratified split completed:")
print(f"Training set: {X_train.shape[0]} recipes")
print(f"Test set:     {X_test.shape[0]} recipes")
print(f"Total:        {X_train.shape[0] + X_test.shape[0]} recipes")

# Verify class ratios are preserved
print(f"\n Class ratio verification (train% → test%):")
for flavor in flavor_categories:
    train_pct = y_train[flavor].mean() * 100
    test_pct = y_test[flavor].mean() * 100
    diff = abs(train_pct - test_pct)
    status = '✅' if diff < 2.0 else '⚠️'
    print(f"{status} {flavor:<12}: {train_pct:5.1f}% → {test_pct:5.1f}%  (Δ {diff:.1f}%)")

print(f"\nFinal dataset shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")


Unique flavor combos: 52
Combos with < 3 samples (bucketed as 'rare'): 20
Stratification groups: 33

Stratified split completed:
Training set: 2153 recipes
Test set:     539 recipes
Total:        2692 recipes

 Class ratio verification (train% → test%):
✅ bittersweet :  18.0% →  18.2%  (Δ 0.2%)
✅ citrus      :  35.6% →  35.4%  (Δ 0.1%)
✅ creamy      :   4.4% →   4.5%  (Δ 0.1%)
✅ floral      :   4.5% →   4.5%  (Δ 0.1%)
✅ fruity      :  27.6% →  27.5%  (Δ 0.2%)
✅ herbal      :  16.3% →  16.9%  (Δ 0.5%)
✅ savoury     :   3.4% →   3.3%  (Δ 0.1%)
✅ spicy       :   5.6% →   5.8%  (Δ 0.2%)
✅ sweet       :   5.5% →   5.4%  (Δ 0.1%)

Final dataset shapes:
X_train: (2153, 242)
X_test:  (539, 242)
y_train: (2153, 9)
y_test:  (539, 9)


In [15]:
# These thresholds were chosen after a lot of experiments
optimal_thresholds = {
    'bittersweet': 0.50,
    'citrus': 0.50,
    'creamy': 0.60,
    'floral': 0.90,
    'fruity': 0.60,
    'herbal': 0.55,
    'savoury': 0.80,
    'spicy': 0.75,
    'sweet': 0.80
}

In [18]:
class CocktailTastePredictor(BaseEstimator, ClassifierMixin):
    def __init__(self, models_dict, thresholds, feature_columns, flavor_categories):
        self.models_dict = models_dict
        self.thresholds = thresholds
        self.feature_columns = feature_columns
        self.flavor_categories = flavor_categories
        
    def preprocess(self, input_data):
        if isinstance(input_data, np.ndarray):
            if input_data.shape[1] != len(self.feature_columns):
                raise ValueError(f"Expected {len(self.feature_columns)} columns, got {input_data.shape[1]}")
            return input_data

        if isinstance(input_data, list):
            return np.array([[item.get(col, 0.0) for col in self.feature_columns] for item in input_data])

        if isinstance(input_data, dict):
            return np.array([[input_data.get(col, 0.0) for col in self.feature_columns]])
            
        elif isinstance(input_data, pd.DataFrame):
            missing_cols = set(self.feature_columns) - set(input_data.columns)
            if missing_cols:
                input_data = pd.concat([input_data, pd.DataFrame(0, index=input_data.index, columns=list(missing_cols))], axis=1)
            return input_data[self.feature_columns].fillna(0.0).values
        else:
            raise ValueError("Input must be a numpy array, list, dict, or pandas DataFrame")

    def predict_proba(self, X):
        features = self.preprocess(X)
        probas = {}
        for flavor in self.flavor_categories:
            probas[flavor] = self.models_dict[flavor].predict_proba(features)[:, 1]
        return probas

    def predict(self, X):
        probas = self.predict_proba(X)
        predictions = {}
        for flavor in self.flavor_categories:
            predictions[flavor] = (probas[flavor] >= self.thresholds[flavor]).astype(int)
        return predictions

In [19]:
trained_models = {}
results = {}

print("Training Logistic Regression models (one per flavor)")
for i, flavor in enumerate(flavor_categories, 1):
    print(f"\n   {i:2}. Training model for: {flavor:<12}")
    
    # Create and train model
    lr_model = LogisticRegression(
        penalty='l1', 
        solver='liblinear', 
        max_iter=1000, 
        random_state=SEED, 
        class_weight='balanced'
    )
    lr_model.fit(X_train, y_train[flavor])
    
    # Store the actual trained model
    trained_models[flavor] = lr_model
    
    # Make predictions on TEST set
    y_test_proba = lr_model.predict_proba(X_test)[:, 1]
    y_pred_optimized = (y_test_proba >= optimal_thresholds[flavor]).astype(int)
    
    # Calculate metrics with the OPTIMIZED threshold
    metrics = {
        'accuracy': accuracy_score(y_test[flavor], y_pred_optimized),
        'precision': precision_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'recall': recall_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'f1': f1_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'auc': roc_auc_score(y_test[flavor], y_test_proba)
    }
    
    # Store feature importance (top ingredients)
    feature_importance = pd.DataFrame({
        'ingredient': ingredient_columns,
        'coefficient': lr_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    results[flavor] = {
        'threshold': optimal_thresholds[flavor],
        'metrics': metrics,
        'top_ingredients': feature_importance.head(5),
    }
    
    # Print metrics beautifully 
    print(f"Performance comparison:")
    print(f"Threshold: ({optimal_thresholds[flavor]:.2f})")
    print(f"AUC:       {metrics['auc']:.3f}")
    print(f"Precision: {metrics['precision']:.3f}")
    print(f"Recall:    {metrics['recall']:.3f}")
    print(f"F1-Score:  {metrics['f1']:.3f}")

print("\n All Logistic Regression models trained successfully!")

Training Logistic Regression models (one per flavor)

    1. Training model for: bittersweet 
Performance comparison:
Threshold: (0.50)
AUC:       0.956
Precision: 0.784
Recall:    0.816
F1-Score:  0.800

    2. Training model for: citrus      
Performance comparison:
Threshold: (0.50)
AUC:       0.893
Precision: 0.701
Recall:    0.859
F1-Score:  0.772

    3. Training model for: creamy      
Performance comparison:
Threshold: (0.60)
AUC:       0.995
Precision: 0.913
Recall:    0.875
F1-Score:  0.894

    4. Training model for: floral      
Performance comparison:
Threshold: (0.90)
AUC:       0.968
Precision: 0.619
Recall:    0.542
F1-Score:  0.578

    5. Training model for: fruity      
Performance comparison:
Threshold: (0.60)
AUC:       0.881
Precision: 0.756
Recall:    0.608
F1-Score:  0.674

    6. Training model for: herbal      
Performance comparison:
Threshold: (0.55)
AUC:       0.821
Precision: 0.451
Recall:    0.604
F1-Score:  0.516

    7. Training model for: savoury     


In [20]:
production_model = CocktailTastePredictor(
    models_dict=trained_models,
    thresholds=optimal_thresholds,
    feature_columns=list(ingredient_columns),
    flavor_categories=flavor_categories
)

os.makedirs("model_output", exist_ok=True)
model_path = "model_output/model.joblib"

joblib.dump(production_model, model_path)

with tarfile.open('model.tar.gz', "w:gz") as tar:
    tar.add(model_path, arcname="model.joblib")

print("\n Production model packaged successfully as 'model.joblib' & 'model.tar.gz'!")


 Production model packaged successfully as 'model.joblib' & 'model.tar.gz'!
